In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

import sklearn
sklearn.set_config(display='text')
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from sklearn.svm import SVC # 서포트 벡터 머신 알고리즘을 사용하기 위해 import 한다.

서포트 벡터 머신(support vector machine)은 벡터를 기준으로 클래스를 판단하는 방식으로 클래스를 구분하는 여러 방법 중 중심선과 경계선을 이용해 데이터를 구분한다. 경계선을 서포트 벡터라고 하며, 이것이 서포트 벡터 머신이라는 이름의 유래이다.

중심선을 그리기 위해서는 중심선에 수직인 벡터 $w$를 구하는 것이 중요하다. 중심선에 수직인 벡터 $w$와 데이터 포인트 $x$를 내적했을 때 내적값 $c$가 되는 지점이 중심헌이 되고 $w^T = c$라고 표시한다.

내적값 $c$가 되는 지점인 중심선을 기준으로 영역을 나눌 수 있다. 데이터 공간을 내적없이 $c$보다 큰 영역과 $c$보다 작은 영역으로 각각 나누면 중심선의 윗부분과 아랫부분을 나눌 수 있다는 것을 알 수 있다.

서포트 벡터 머신은 마진(margin)을 최대화 하는 것이 목적이다. 마진이란 서포트 벡터간 너비를 의미한다.

소프트 마진(soft magine)  
서포트 벡터 머신은 데이가 잘못 분류되는 경우를 고려하지 않는다. 하지만 잘못 분류된 데이터가 하나도 없다는 것은 현실적으로 너무 엄격한 기준이므로 성립되기 여렵다. 소프트 마진은 기존 서포트 벡터 머신의 완화해서 잘못 분류된 데이터도 어느 정도 허용하는 방법이다.

커널 서포트 벡터 머신(kernel support vector machine)  
커널 서포트 벡터 머신이란 피쳐 공간을 변형한 후 서포트 벡터 머신을 적용하는 것을 의미한다.  
좌표 평면을 빳빳한 종이라 생각하고 종이 위에 데이터가 펴져있다고 가정하면 종이를 구부렸을 때 기존 좌표 공간과 구부러진 좌표 공간의 데이터 좌표가 서로 다를 것이다. 구부러진 공간에 서포트 벡터 머신을 적용한 후 종이를 펴면 데이터가 잘 분리되는 것을 볼 수 있다.

와인 데이터를 사용해서 와인 종류를 분류하는 모델을 생성하고 학습시킨다.

In [5]:
# 데이터 불러오기
raw_data = datasets.load_wine() # 사이킷런 라이브러리가 제공하는 와인 데이터를 불러온다.
# print(raw_data)

# 피쳐, 레이블 데이터 저장
xData = raw_data.data # 피쳐 데이터를 저장한다.
yData = raw_data.target # 피쳐 데이터에 따른 레이블을 저장한다.
# print(xData.shape, yData.shape)

# 학습 데이터와 테스트 데이터로 분할
x_train, x_test, y_train, y_test = train_test_split(xData, yData, random_state=0)
# print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

# 데이터 표준화(정규화)
scaler = StandardScaler() # 표준화 스케일러 객체를 만든다.
x_train = scaler.fit_transform(x_train) # 학습 데이터를 표준화 스케일러로 표준화하고 적용한다.
x_test = scaler.transform(x_test) # 테스트 데이터를 학습 데이터로 표준화한 스케일러에 적용한다.

# 모델 생성 후 데이터 학습
# 서포트 벡터 머신 모델의 kernel 속성으로 커널 함수 종류를 지정해서 모델을 만든다.
# kernel 속성의 기본값은 'rbf'(가우시안 커널, 방사 기저 함수)이고 'linear'(선형), 'poly'(다항식), 'sidmoid' 중에서 지정이 가능하다.
# 학습된 모델로 테스트 데이터를 예측할 때 predict() 메소드를 사용해서 예측을 레이블 값으로 할 때는 아무 문제가 없지만 predict_proba() 메소드를 사용해서 예측을
# 값이 아닌 비율로 예측하면 에러가 발생된다.
# predict_proba() 메소드를 서포트 벡터 머신 모델에 사용하려면 probability 속성의 속성값을 True로 지정해야 한다.
model = SVC(kernel='rbf', probability=True).fit(x_train, y_train) # 서포트 벡터 머신 모델을 만들고 학습시킨다.

학습된 모델로 테스트 데이터를 예측한다.

In [6]:
predict = model.predict(x_test) # predict() 메소드의 인수로 표준화된 테스트 데이터(x_test)를 넘겨서 서포트 벡터 머신 모델을 예측한다.
print(predict)

[0 2 1 0 1 1 0 2 1 1 2 2 0 1 2 1 0 0 1 0 1 0 0 1 1 1 1 1 1 2 0 0 1 0 0 0 2
 1 1 2 0 0 1 1 1]


In [7]:
predict_proba = model.predict_proba(x_test) # predict_proba() 메소드의 인수로 표준화된 테스트 데이터(x_test)를 넘겨서 각 클래스에 속할 확률로 예측한다.
print(predict_proba)

[[0.98973856 0.00390393 0.00635751]
 [0.01211313 0.01858889 0.96929798]
 [0.01646974 0.97328254 0.01024772]
 [0.9662623  0.02483371 0.00890399]
 [0.12400874 0.83832653 0.03766473]
 [0.13068316 0.8197445  0.04957234]
 [0.98704919 0.00372525 0.00922556]
 [0.00766473 0.00896722 0.98336804]
 [0.00818937 0.98892972 0.00288091]
 [0.00358956 0.98829182 0.00811862]
 [0.0211084  0.07594662 0.90294498]
 [0.02477488 0.10324775 0.87197737]
 [0.99245807 0.00261232 0.00492961]
 [0.09542275 0.89140819 0.01316905]
 [0.00860552 0.01031538 0.9810791 ]
 [0.00245006 0.99466283 0.00288711]
 [0.87050713 0.10515777 0.0243351 ]
 [0.94603407 0.03185451 0.02211141]
 [0.01015296 0.78392683 0.20592021]
 [0.99132711 0.00305357 0.00561933]
 [0.20717699 0.76746795 0.02535507]
 [0.94538822 0.04057474 0.01403704]
 [0.89112898 0.09832295 0.01054807]
 [0.02025438 0.97690757 0.00283805]
 [0.00929111 0.95332036 0.03738853]
 [0.00478799 0.99160015 0.00361185]
 [0.01181003 0.97858746 0.00960251]
 [0.00168699 0.99571391 0.00

학습된 모델을 평가한다.

In [8]:
# 혼동 행렬
# confusion_matrix() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 혼동 행렬을 출력한다.
confusion = confusion_matrix(y_test, predict)
print(confusion)

[[16  0  0]
 [ 0 21  0]
 [ 0  0  8]]


In [9]:
# 분류 리포트
# classification_report() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 분류 리포트를 출력한다.
classification = classification_report(y_test, predict, target_names=raw_data.target_names)
print(classification)

              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        16
     class_1       1.00      1.00      1.00        21
     class_2       1.00      1.00      1.00         8

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45



In [10]:
# 정확도 평가
# accuracy_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 정확도를 계산한다.
accuracy = accuracy_score(y_test, predict)
print(accuracy)

1.0


In [11]:
# 정밀도 평가
# precision_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 정밀도를 계산한다.
precision = precision_score(y_test, predict, average=None)
print(precision)

[1. 1. 1.]


In [12]:
# 재현율 평가
# recall_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 재현율을 계산한다.
recall = recall_score(y_test, predict, average=None)
print(recall)

[1. 1. 1.]


In [13]:
# f1 score 평가
# f1_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 f1 score를 계산한다.
f1 = f1_score(y_test, predict, average=None)
print(f1)

[1. 1. 1.]
